# Setup From GitHub

Set `REPO_URL` to your GitHub repo. These notebooks avoid reinstalling PyTorch because Kaggle already ships a CUDA-compatible build.

In [ ]:
REPO_URL = "https://github.com/ahmad903501/qwen-dpo-alignment.git"
BRANCH = "main"
REPO_DIR = "/kaggle/working/qwen-dpo-alignment"


In [ ]:
from pathlib import Path

repo_path = Path(REPO_DIR)
if not repo_path.exists():
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull --ff-only origin {BRANCH}

%cd {REPO_DIR}
!pwd


In [ ]:
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors
!python -m pip install -q -e . --no-deps

import torch, transformers, datasets
print("torch", torch.__version__)
print("transformers", transformers.__version__)
print("datasets", datasets.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA devices:", torch.cuda.device_count())
!nvidia-smi

import os
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
except Exception as exc:
    print(f"No Kaggle HF_TOKEN secret loaded: {exc}")

if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"])



# Evaluation / Judging Only

Attach the generation notebook outputs as a Kaggle input dataset, then set `EVAL_DIR`. This notebook does not train or generate model outputs.

In [ ]:
from pathlib import Path
import zipfile

SUBSET_SIZE = 300
SEED = 42
MODEL_RUN_NAME = "model"
ARCHIVE_BASENAME = f"{MODEL_RUN_NAME}_alignment_eval_outputs_{SUBSET_SIZE}_seed{SEED}"

# Supported local layouts, checked in order.
# `archive/` is useful when you manually unzip/download eval files into the repo.
# `outputs/eval/` is useful when generation and evaluation run in the same Kaggle session.
LOCAL_EVAL_DIRS = [Path("archive"), Path("outputs/eval")]

# If using an attached Kaggle dataset, set this to one of:
#   /kaggle/input/YOUR_DATASET/archive
#   /kaggle/input/YOUR_DATASET/eval
#   /kaggle/input/YOUR_DATASET/model_alignment_eval_outputs_300_seed42.zip
# It may also point to any directory containing dpo_vs_sft_pairs.jsonl.
ATTACHED_EVAL_PATH = None

JUDGE_MODEL = "google/gemini-3.6-flash"
JUDGE_BATCH_SIZE = 5
JUDGE_SLEEP_SECONDS = 5
JUDGE_MAX_RETRIES = 3
REUSE_JUDGMENTS = True


In [ ]:
REQUIRED_PAIR_FILE = "dpo_vs_sft_pairs.jsonl"

def directory_has_eval_files(path):
    return path.is_dir() and (path / REQUIRED_PAIR_FILE).is_file()

def extract_eval_zip(zip_path):
    extract_root = Path("/kaggle/working/imported_eval_outputs")
    extract_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(extract_root)

    candidates = [extract_root / "eval", extract_root / "archive", extract_root]
    for candidate in candidates:
        if directory_has_eval_files(candidate):
            return candidate

    pair_matches = sorted(extract_root.glob(f"**/{REQUIRED_PAIR_FILE}"))
    if pair_matches:
        return pair_matches[0].parent
    raise FileNotFoundError(f"Could not find {REQUIRED_PAIR_FILE} inside {zip_path}")

def resolve_eval_dir():
    if ATTACHED_EVAL_PATH:
        attached = Path(ATTACHED_EVAL_PATH)
        if directory_has_eval_files(attached):
            return attached
        if attached.is_dir():
            pair_matches = sorted(attached.glob(f"**/{REQUIRED_PAIR_FILE}"))
            if pair_matches:
                return pair_matches[0].parent
        if attached.is_file() and attached.suffix == ".zip":
            return extract_eval_zip(attached)
        raise FileNotFoundError(f"ATTACHED_EVAL_PATH does not contain {REQUIRED_PAIR_FILE}: {attached}")

    for local_dir in LOCAL_EVAL_DIRS:
        if directory_has_eval_files(local_dir):
            return local_dir

    archive_matches = sorted(Path("/kaggle/input").glob(f"**/{ARCHIVE_BASENAME}.zip"))
    if archive_matches:
        print(f"Found attached eval archive: {archive_matches[0]}")
        return extract_eval_zip(archive_matches[0])

    pair_matches = sorted(Path("/kaggle/input").glob(f"**/{REQUIRED_PAIR_FILE}"))
    if pair_matches:
        print(f"Found attached eval files: {pair_matches[0].parent}")
        return pair_matches[0].parent

    raise FileNotFoundError(
        f"Could not find {REQUIRED_PAIR_FILE}. Put eval files in archive/, outputs/eval/, "
        "or set ATTACHED_EVAL_PATH to a directory/zip containing the pairwise files."
    )

EVAL_DIR = resolve_eval_dir()
print("Using EVAL_DIR:", EVAL_DIR)


## Inspect Pairwise Files

In [ ]:
!ls -lah {EVAL_DIR}
!sed -n '1,2p' {EVAL_DIR}/dpo_vs_sft_pairs.jsonl


## Kaggle Model-Proxy Judge

This uses Kaggle Models via `kaggle_benchmarks`, so it does not load a judge model into GPU memory. Judgments are checkpointed and can be resumed.


In [ ]:
!python scripts/judge_pairwise_kaggle.py \
  --pairs_path {EVAL_DIR}/dpo_vs_sft_pairs.jsonl \
  --output_path outputs/eval/dpo_vs_sft_judged_kaggle.jsonl \
  --judge_model {JUDGE_MODEL} \
  --batch_size {JUDGE_BATCH_SIZE} \
  --sleep_seconds {JUDGE_SLEEP_SECONDS} \
  --max_retries {JUDGE_MAX_RETRIES} \
  --reuse_judgments

!python scripts/judge_pairwise_kaggle.py \
  --pairs_path {EVAL_DIR}/dpo_vs_base_pairs.jsonl \
  --output_path outputs/eval/dpo_vs_base_judged_kaggle.jsonl \
  --judge_model {JUDGE_MODEL} \
  --batch_size {JUDGE_BATCH_SIZE} \
  --sleep_seconds {JUDGE_SLEEP_SECONDS} \
  --max_retries {JUDGE_MAX_RETRIES} \
  --reuse_judgments

!python scripts/judge_pairwise_kaggle.py \
  --pairs_path {EVAL_DIR}/sft_vs_base_pairs.jsonl \
  --output_path outputs/eval/sft_vs_base_judged_kaggle.jsonl \
  --judge_model {JUDGE_MODEL} \
  --batch_size {JUDGE_BATCH_SIZE} \
  --sleep_seconds {JUDGE_SLEEP_SECONDS} \
  --max_retries {JUDGE_MAX_RETRIES} \
  --reuse_judgments


## Artifact List

In [ ]:
!find outputs/eval -maxdepth 1 -type f 2>/dev/null | sort || true
